# 04.03.03 YOLO Model Training

**Phase 04.03 — YOLO Document Detection**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Formulate the training pipeline, hyperparameter configuration, and experiment tracking for YOLO document region detection.


### Research Objectives

1. Configure YOLO model architecture (e.g., YOLO11s / YOLO11s) for document layout analysis.
2. Define training hyperparameters: image size (e.g., 640 or 1024), epochs, batch size, learning rate schedule, and augmentations.
3. Configure checkpointing, early stopping, and TensorBoard / experiment logging.
4. Establish training execution harness (ready for run upon dataset authorization).


### Inputs & Expected Outputs

**Required Inputs:**
- `Validated YOLO dataset (`data/processed/yolo_dataset/data.yaml`).`
- `Pretrained YOLO backbone weights specification.`

**Expected Outputs:**
- `Trained model destination: `models/trained/yolo/document_detector_best.pt`.`
- `Training configuration and hyperparameters log.`


### 1. Environment Setup & Library Imports

In [10]:
import os
import sys
from pathlib import Path
import yaml
import torch

def _find_root(p=Path.cwd()):
    for parent in [p, *p.parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths

paths = DocumentProcessingPaths(PROJECT_ROOT)
print(f"Project root: {paths.root}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


Project root: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2
CUDA Available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU


### 1. Setup Environment and Hyperparameters

In [11]:
TRAIN_OUTPUT_DIR = paths.yolo_models_dir
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML = paths.yolo_dataset_dir / "data.yaml"

# Standardized training configuration for Ultralytics YOLO11s
training_config = {
    'model': 'yolo11s.pt',              # Architecture: YOLO11s Nano detection model
    'data': str(DATA_YAML.as_posix()),  # Path to dataset data.yaml
    'epochs': 50,                       # Configurable epoch budget
    'imgsz': 640,                       # Standard input resolution for documents
    'batch': 16 if 'torch' in sys.modules and getattr(sys.modules['torch'], 'cuda', None) and sys.modules['torch'].cuda.is_available() else 4,
    'device': 0 if 'torch' in sys.modules and getattr(sys.modules['torch'], 'cuda', None) and sys.modules['torch'].cuda.is_available() else 'cpu',
    'project': str(TRAIN_OUTPUT_DIR.as_posix()),
    'name': 'ofphil_yolo_detector',
    'seed': 20261123,                   # Strict reproducible research seed
    'deterministic': True,              # Enforce deterministic cuDNN algorithms
    'patience': 15,                     # Early stopping patience
    'save': True,                       # Save training checkpoints and final weights
    'save_period': 5,                   # Save checkpoint every 5 epochs
    'workers': 4,                       # Dataloader workers
    'pretrained': True                  # Start from pretrained ImageNet/COCO weights
}

with open(TRAIN_OUTPUT_DIR / 'training_hyperparameters.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(training_config, f, default_flow_style=False)
print(f"Training configuration saved -> {TRAIN_OUTPUT_DIR / 'training_hyperparameters.yaml'}")


Training configuration saved -> C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\models\trained\yolo\training_hyperparameters.yaml


### 2. Training Command / API Formulation

In [12]:
# Executable training routine using Ultralytics YOLO API
# Requires manual execution by the researcher in JupyterLab
def execute_yolo_training(config):
    try:
        from ultralytics import YOLO
        print(f"Initializing Ultralytics YOLO model: {config['model']}")
        model = YOLO(config['model'])
        print(f"Starting training with seed {config.get('seed', 20261123)} on OFPHIL document layout dataset...")
        results = model.train(
            data=config['data'],
            epochs=config['epochs'],
            imgsz=config['imgsz'],
            batch=config['batch'],
            device=config['device'],
            project=config['project'],
            name=config['name'],
            seed=config.get('seed', 20261123),
            deterministic=config.get('deterministic', True),
            patience=config.get('patience', 15),
            save=config.get('save', True),
            save_period=config.get('save_period', 5),
            workers=config.get('workers', 4)
        )
        print("Training completed successfully.")
        print(f"Best model saved at: {TRAIN_OUTPUT_DIR / config['name'] / 'weights' / 'best.pt'}")
        print(f"Last model saved at: {TRAIN_OUTPUT_DIR / config['name'] / 'weights' / 'last.pt'}")
        return results
    except ImportError:
        print("Ultralytics package not detected in current environment.")
        print("Install required packages via: pip install ultralytics torch torchvision")
        return None

print("Training routine defined for YOLO11s. Ready for manual execution.")


Training routine defined for YOLOv8n. Ready for manual execution.


### 2.1 Execute YOLO11s Training

Run the cell below in JupyterLab to start fine-tuning `yolo11s.pt` on the OFPHIL dataset. This will train for 50 epochs and save checkpoints to `models/trained/yolo/ofphil_yolo_detector/weights/`.

In [13]:
# Trigger training run
training_results = execute_yolo_training(training_config)


Initializing Ultralytics YOLO model: yolov8n.pt
Starting training with seed 20261123 on OFPHIL document layout dataset...
New https://pypi.org/project/ultralytics/8.4.218 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.216  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/ASUS TUF/Desktop/Project in Program/Thesis Project/ofphil-classification-v2/data/processed/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5,

### 3. Checkpoint and Metric Logging Setup

In [14]:
# Check if trained weights already exist from prior runs
best_weights = TRAIN_OUTPUT_DIR / 'ofphil_yolo_detector' / 'weights' / 'best.pt'
if best_weights.exists():
    print(f"Found existing trained checkpoint: {best_weights} ({best_weights.stat().st_size / (1024*1024):.2f} MB)")
else:
    print("No trained checkpoint found yet. Run the training cell above to generate best.pt.")


No trained checkpoint found yet. Run the training cell above to generate best.pt.


### Findings & Observations

- **Observations:** Document hyperparameter rationale (choice of image resolution, augmentation constraints for document text preservation).
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Training hyperparameters documented in structured config.
- [ ] Input paths verified against data.yaml.
- [ ] Output checkpoint directory prepared.
- [ ] No unapproved training execution triggered.
